**Rihana Nicolas — Labos Benjamin — Groupe 4**


# Exploratory Data Analysis with Pyspark and Spark SQL

The following notebook utilizes New York City taxi data from [TLC Trip Record Data](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page)

## Instructions

- Load and explore nyc taxi data from january 0f 2019. The exercises can be executed using pyspark or spark sql (a subset of the questions will be re-answered using the language not chosen for the  main work).
- Load the zone lookup table to answer the questions about the nyc boroughs.  
- Load nyc taxi data from January of 2025 and compare data.  
- With any remaining time, work on the where to go from here section.
- Note: the initial lab is opened as read only. To save work completed utilize the `save notebook as` option and give the lab a new name.

In [ ]:
import os
import sys
import glob
import requests

# Ensure the Spark Python package included in the Docker image is importable.
# This keeps the notebook compatible with the current jupyter/pyspark-notebook image.
spark_home = os.environ.get("SPARK_HOME", "/usr/local/spark")
spark_python = os.path.join(spark_home, "python")
py4j_candidates = glob.glob(os.path.join(spark_python, "lib", "py4j-*-src.zip"))
for path in [spark_python] + py4j_candidates[:1]:
    if path not in sys.path:
        sys.path.insert(0, path)

# start a spark session and create a spark context
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("nyc_taxi") \
    .getOrCreate()

sc = spark.sparkContext


In [ ]:
# set dl url for January 2019 trip data
download_url = 'https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2019-01.parquet'

# get the data
response = requests.get(download_url)

# check that response was good and save the data
jan_2019_trip_data = "yellow_tripdata_2019-01.parquet"
if response.status_code == 200:
    # Changed 'response' to 'data' here
    with open(jan_2019_trip_data, "wb") as f:
        f.write(response.content)


In [ ]:
# create the dataframe
df_trips = spark.read.parquet(jan_2019_trip_data)

# A brief note on handling data sources in spark

The command above works well for loading data from parquet files because parquet is a self descibing file format, meaning that the metadata needed to build the dataframe is included directly in the format. However, when working with other formats such as csv or json, a schema must be provided or infered. In production code the schema should always be explicitly provided but during the data exploration phase it is acceptable to infer the schema, and when infering the schema it often best to use `.option("samplingRatio", <small-portion-of-data>)` to avoid using the entire dataset for schema inference.

```python
df_trips = spark.read.format("csv") \
    .option("header", "true") \
    .option("sep", ",") \
    .option("samplingRatio", 0.01) \
    .load("large_dataset.csv")
```

In [ ]:
# Show the dataframe
df_trips.show()

## Lab

The main analysis below uses **PySpark DataFrames**.  
For time-based questions, the analysis is restricted to pickups occurring in **January 2019** so that invalid timestamps outside the target month do not distort the results.


In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Add a unique identifier for each trip
df_trips = df_trips.withColumn("trip_id", F.monotonically_increasing_id())

# Add useful time columns
df_2019 = (
    df_trips
    .withColumn("pickup_ts", F.col("tpep_pickup_datetime"))
    .withColumn("dropoff_ts", F.col("tpep_dropoff_datetime"))
    .withColumn(
        "trip_duration_min",
        F.expr("timestampdiff(SECOND, pickup_ts, dropoff_ts) / 60.0")
    )
    .withColumn("trip_date", F.to_date("pickup_ts"))
    .withColumn("pickup_hour", F.hour("pickup_ts"))
    .withColumn("day_of_week", F.date_format("pickup_ts", "EEEE"))
)

# Keep January 2019 for analyses based on calendar date/time
df_jan = df_2019.filter(
    (F.col("pickup_ts") >= F.lit("2019-01-01 00:00:00")) &
    (F.col("pickup_ts") < F.lit("2019-02-01 00:00:00"))
)

print("Rows in original file:", df_trips.count())
print("Rows with pickup in January 2019:", df_jan.count())


### Part 1 — DataFrame analysis


In [ ]:
# Trip with the highest passenger count
df_jan.select(
    "trip_id", "tpep_pickup_datetime", "tpep_dropoff_datetime",
    "passenger_count", "trip_distance", "fare_amount", "tip_amount"
).orderBy(F.desc("passenger_count")).show(1, truncate=False)

# Average passenger count
df_jan.select(
    F.round(F.avg("passenger_count"), 3).alias("average_passenger_count")
).show()


In [ ]:
# Shortest / longest trip by distance (positive distances only)
valid_distance = df_jan.filter(F.col("trip_distance") > 0)

print("Shortest positive trip by distance:")
valid_distance.select(
    "trip_id", "trip_distance", "tpep_pickup_datetime", "tpep_dropoff_datetime"
).orderBy(F.asc("trip_distance")).show(1, truncate=False)

print("Longest trip by distance:")
valid_distance.select(
    "trip_id", "trip_distance", "tpep_pickup_datetime", "tpep_dropoff_datetime"
).orderBy(F.desc("trip_distance")).show(1, truncate=False)

# Shortest / longest trip by duration (positive durations only)
valid_duration = df_jan.filter(F.col("trip_duration_min") > 0)

print("Shortest positive trip by time:")
valid_duration.select(
    "trip_id", "trip_duration_min", "tpep_pickup_datetime", "tpep_dropoff_datetime"
).orderBy(F.asc("trip_duration_min")).show(1, truncate=False)

print("Longest trip by time:")
valid_duration.select(
    "trip_id", "trip_duration_min", "tpep_pickup_datetime", "tpep_dropoff_datetime"
).orderBy(F.desc("trip_duration_min")).show(1, truncate=False)


In [ ]:
# Busiest and slowest single day
daily_counts = (
    df_jan.groupBy("trip_date")
    .count()
    .orderBy(F.desc("count"))
)

print("Busiest day:")
daily_counts.show(1)

print("Slowest day:")
daily_counts.orderBy(F.asc("count")).show(1)


In [ ]:
# Busiest and slowest time of day, bucketed by pickup hour
hourly_counts = df_jan.groupBy("pickup_hour").count()

print("Busiest pickup hour:")
hourly_counts.orderBy(F.desc("count")).show(1)

print("Slowest pickup hour:")
hourly_counts.orderBy(F.asc("count")).show(1)


In [ ]:
# Average traffic by day of week
# First count trips per calendar day, then average those daily totals by weekday.
weekday_average = (
    df_jan.groupBy("trip_date", "day_of_week").count()
    .groupBy("day_of_week")
    .agg(F.round(F.avg("count"), 2).alias("avg_trips_per_day"))
    .orderBy(F.desc("avg_trips_per_day"))
)

weekday_average.show(7, truncate=False)

print("Busiest weekday on average:")
weekday_average.show(1, truncate=False)

print("Slowest weekday on average:")
weekday_average.orderBy(F.asc("avg_trips_per_day")).show(1, truncate=False)


In [ ]:
# Relationship between trip distance / passenger count and tip amount
tip_analysis = df_jan.filter(
    (F.col("trip_distance") >= 0) &
    (F.col("passenger_count") >= 0) &
    (F.col("tip_amount") >= 0)
)

tip_analysis.select(
    F.round(F.corr("trip_distance", "tip_amount"), 4).alias("corr_distance_tip"),
    F.round(F.corr("passenger_count", "tip_amount"), 4).alias("corr_passengers_tip")
).show()

print("Average tip by passenger count:")
tip_analysis.groupBy("passenger_count").agg(
    F.count("*").alias("trips"),
    F.round(F.avg("tip_amount"), 2).alias("avg_tip")
).orderBy("passenger_count").show(20)


In [ ]:
# Highest extra charge and the corresponding trip
df_jan.select(
    "trip_id", "tpep_pickup_datetime", "tpep_dropoff_datetime",
    "extra", "fare_amount", "total_amount"
).orderBy(F.desc("extra")).show(1, truncate=False)


In [ ]:
# Quick outlier checks on the original January data
df_jan.select(
    F.min("trip_distance").alias("min_distance"),
    F.max("trip_distance").alias("max_distance"),
    F.min("fare_amount").alias("min_fare"),
    F.max("fare_amount").alias("max_fare"),
    F.min("trip_duration_min").alias("min_duration_min"),
    F.max("trip_duration_min").alias("max_duration_min"),
    F.max("passenger_count").alias("max_passengers"),
    F.max("tip_amount").alias("max_tip")
).show(truncate=False)

df_jan.select(
    F.sum(F.when(F.col("trip_distance") < 0, 1).otherwise(0)).alias("negative_distance_rows"),
    F.sum(F.when(F.col("fare_amount") < 0, 1).otherwise(0)).alias("negative_fare_rows"),
    F.sum(F.when(F.col("trip_duration_min") <= 0, 1).otherwise(0)).alias("non_positive_duration_rows")
).show()


**Outlier reasoning.** Taxi trips should not normally have negative fares, negative distances, or a drop-off time earlier than the pickup time. Extremely large distances, durations, passenger counts, fares or tips should also be treated cautiously. The checks above expose those values rather than silently deleting them. For the shortest-trip questions, only positive distance/duration values were used so that invalid or zero-length records do not become the answer.


### Part 2 — Enriching the data with the taxi-zone lookup


In [ ]:
# Download the taxi zone lookup table
zone_url = "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv"
zone_response = requests.get(zone_url)
zone_file = "taxi_zone_lookup.csv"

if zone_response.status_code == 200:
    with open(zone_file, "wb") as f:
        f.write(zone_response.content)
else:
    raise RuntimeError(f"Zone lookup download failed: HTTP {zone_response.status_code}")

zones = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(zone_file)
)

zones.show(5, truncate=False)


In [ ]:
# Join the lookup twice: once for pickup zone and once for dropoff zone
pickup_zones = zones.select(
    F.col("LocationID").alias("PULocationID"),
    F.col("Borough").alias("pickup_borough"),
    F.col("Zone").alias("pickup_zone")
)

dropoff_zones = zones.select(
    F.col("LocationID").alias("DOLocationID"),
    F.col("Borough").alias("dropoff_borough"),
    F.col("Zone").alias("dropoff_zone")
)

df_enriched = (
    df_jan
    .join(pickup_zones, on="PULocationID", how="left")
    .join(dropoff_zones, on="DOLocationID", how="left")
)

df_enriched.select(
    "trip_id", "pickup_borough", "pickup_zone",
    "dropoff_borough", "dropoff_zone"
).show(5, truncate=False)


In [ ]:
# Borough with the most pickups and dropoffs
print("Most pickups by borough:")
df_enriched.groupBy("pickup_borough").count().orderBy(F.desc("count")).show()

print("Most dropoffs by borough:")
df_enriched.groupBy("dropoff_borough").count().orderBy(F.desc("count")).show()


In [ ]:
# Busiest and slowest pickup hour by borough
borough_hour_counts = (
    df_enriched
    .filter(F.col("pickup_borough").isNotNull())
    .groupBy("pickup_borough", "pickup_hour")
    .count()
)

w_busy = Window.partitionBy("pickup_borough").orderBy(F.desc("count"), F.asc("pickup_hour"))
w_slow = Window.partitionBy("pickup_borough").orderBy(F.asc("count"), F.asc("pickup_hour"))

busy_hours = borough_hour_counts.withColumn("rn", F.row_number().over(w_busy)).filter("rn = 1")
slow_hours = borough_hour_counts.withColumn("rn", F.row_number().over(w_slow)).filter("rn = 1")

print("Busiest hour by borough:")
busy_hours.select("pickup_borough", "pickup_hour", "count").orderBy("pickup_borough").show(truncate=False)

print("Slowest observed hour by borough:")
slow_hours.select("pickup_borough", "pickup_hour", "count").orderBy("pickup_borough").show(truncate=False)


In [ ]:
# Busiest day of the week by borough
borough_weekday_counts = (
    df_enriched
    .filter(F.col("pickup_borough").isNotNull())
    .groupBy("pickup_borough", "day_of_week")
    .count()
)

w_day = Window.partitionBy("pickup_borough").orderBy(F.desc("count"), F.asc("day_of_week"))

borough_weekday_counts.withColumn(
    "rn", F.row_number().over(w_day)
).filter("rn = 1").select(
    "pickup_borough", "day_of_week", "count"
).orderBy("pickup_borough").show(truncate=False)


In [ ]:
# Average trip distance and average fare by pickup borough
borough_averages = (
    df_enriched
    .filter(F.col("pickup_borough").isNotNull())
    .groupBy("pickup_borough")
    .agg(
        F.round(F.avg("trip_distance"), 2).alias("avg_trip_distance"),
        F.round(F.avg("fare_amount"), 2).alias("avg_fare_amount")
    )
    .orderBy("pickup_borough")
)

borough_averages.show(truncate=False)


In [ ]:
# Highest and lowest non-negative fare, with associated pickup borough
valid_fares = df_enriched.filter(F.col("fare_amount") >= 0)

print("Highest fare:")
valid_fares.select(
    "trip_id", "fare_amount", "pickup_borough", "pickup_zone",
    "tpep_pickup_datetime", "tpep_dropoff_datetime"
).orderBy(F.desc("fare_amount")).show(1, truncate=False)

print("Lowest non-negative fare:")
valid_fares.select(
    "trip_id", "fare_amount", "pickup_borough", "pickup_zone",
    "tpep_pickup_datetime", "tpep_dropoff_datetime"
).orderBy(F.asc("fare_amount")).show(1, truncate=False)


In [ ]:
# Load January 2025 and compare average metrics with January 2019
download_url_2025 = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-01.parquet"
response_2025 = requests.get(download_url_2025)
jan_2025_trip_data = "yellow_tripdata_2025-01.parquet"

if response_2025.status_code == 200:
    with open(jan_2025_trip_data, "wb") as f:
        f.write(response_2025.content)
else:
    raise RuntimeError(f"2025 trip download failed: HTTP {response_2025.status_code}")

df_2025_raw = spark.read.parquet(jan_2025_trip_data)

df_2025 = (
    df_2025_raw
    .withColumn("trip_duration_min",
        F.expr("timestampdiff(SECOND, tpep_pickup_datetime, tpep_dropoff_datetime) / 60.0")
    )
    .filter(
        (F.col("tpep_pickup_datetime") >= F.lit("2025-01-01 00:00:00")) &
        (F.col("tpep_pickup_datetime") < F.lit("2025-02-01 00:00:00"))
    )
)

def average_metrics(df, year):
    return df.agg(
        F.round(F.avg("trip_distance"), 3).alias("avg_trip_distance"),
        F.round(F.avg("passenger_count"), 3).alias("avg_passenger_count"),
        F.round(F.avg("fare_amount"), 3).alias("avg_fare_amount"),
        F.round(F.avg("tip_amount"), 3).alias("avg_tip_amount"),
        F.round(F.avg("trip_duration_min"), 3).alias("avg_trip_duration_min")
    ).withColumn("year", F.lit(year))

comparison = average_metrics(df_jan, 2019).unionByName(average_metrics(df_2025, 2025))
comparison.select(
    "year", "avg_trip_distance", "avg_passenger_count",
    "avg_fare_amount", "avg_tip_amount", "avg_trip_duration_min"
).show(truncate=False)


The table above gives the requested 2019 vs. 2025 comparison using the same five average metrics. Differences should be interpreted as descriptive changes in the recorded January taxi trips; this simple comparison does not by itself establish why the changes occurred.


### Part 3 — Three questions repeated in Spark SQL


In [ ]:
# Register the enriched DataFrame as a SQL view
df_enriched.createOrReplaceTempView("taxi_2019")


In [ ]:
# SQL question 1: busiest single day
spark.sql("""
    SELECT
        trip_date,
        COUNT(*) AS trips
    FROM taxi_2019
    GROUP BY trip_date
    ORDER BY trips DESC
    LIMIT 1
""").show()


In [ ]:
# SQL question 2: average passenger count
spark.sql("""
    SELECT ROUND(AVG(passenger_count), 3) AS average_passenger_count
    FROM taxi_2019
""").show()


In [ ]:
# SQL question 3 (involves the join result): borough with the most pickups
spark.sql("""
    SELECT
        pickup_borough,
        COUNT(*) AS pickups
    FROM taxi_2019
    WHERE pickup_borough IS NOT NULL
    GROUP BY pickup_borough
    ORDER BY pickups DESC
    LIMIT 1
""").show()


## Conclusion

The lab used DataFrame operations such as `select`, `filter`, `groupBy`, aggregations, window functions and joins, then repeated three questions with Spark SQL. The zone lookup enriched the trip records with borough information, and January 2025 was loaded to compare the same average metrics with January 2019.
